In [3]:
from datasets import load_dataset
import soundfile as sf
import os
import pandas as pd
import shutil
from pathlib import Path
from datetime import datetime


data = load_dataset("brunopbb/ufcg-labmet-fala-texto-main-final")


base_dir = Path("/home/bruno-dev/fala_texto_share/projeto-fala-texto/IA/codigos/utils")
backup_root_dir = Path("/home/bruno-dev/fala_texto_share/projeto-fala-texto/backups")


timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
backup_dir = backup_root_dir / f"backup_{timestamp}"


audio_dir = base_dir / "dataset_backup/audio"
transcription_file = base_dir / "dataset_backup/transcriptions.csv"


audio_dir.mkdir(parents=True, exist_ok=True)
backup_dir.mkdir(parents=True, exist_ok=True)


def save_audio_and_transcriptions(dataset, audio_dir):
    """ Salva os arquivos de áudio e cria um CSV com as transcrições """
    audio_files = []
    transcriptions = []

    for idx, sample in enumerate(dataset):
        audio_filename = f"sample_{idx:05d}.wav"
        audio_path = audio_dir / audio_filename

        sf.write(str(audio_path), sample["audio"]["array"], samplerate=sample["audio"]["sampling_rate"])

        audio_files.append(str(audio_filename))
        transcriptions.append(sample["transcription"])

    df = pd.DataFrame({
        "audio_file": audio_files,
        "transcription": transcriptions
    })

    return df


train_df = save_audio_and_transcriptions(data["train"], audio_dir)
train_df.to_csv(transcription_file, index=False)


test_audio_dir = base_dir / "dataset_backup/test_audio"
test_transcription_file = base_dir / "dataset_backup/test_transcriptions.csv"


test_audio_dir.mkdir(parents=True, exist_ok=True)


test_df = save_audio_and_transcriptions(data["test"], test_audio_dir)
test_df.to_csv(test_transcription_file, index=False)


shutil.copytree(audio_dir, backup_dir / "audio", dirs_exist_ok=True)
shutil.copytree(test_audio_dir, backup_dir / "test_audio", dirs_exist_ok=True)
shutil.copy(transcription_file, backup_dir / "transcriptions.csv")
shutil.copy(test_transcription_file, backup_dir / "test_transcriptions.csv")


shutil.rmtree(audio_dir)
shutil.rmtree(test_audio_dir)
os.remove(transcription_file)
os.remove(test_transcription_file)


huggingface_datasets_dir = Path("huggingface_datasets")
if huggingface_datasets_dir.exists():
    shutil.move(str(huggingface_datasets_dir), str(backup_dir / "huggingface_datasets"))


print(f"Arquivos de áudio salvos em: {backup_dir / 'audio'}")
print(f"Arquivo de transcrições salvo em: {backup_dir / 'transcriptions.csv'}")
print(f"Arquivos de áudio do teste salvos em: {backup_dir / 'test_audio'}")
print(f"Arquivo de transcrições do teste salvo em: {backup_dir / 'test_transcriptions.csv'}")
print(f"Backup completo salvo em: {backup_dir}")
print(f"Pasta 'huggingface_datasets' movida para: {backup_dir / 'huggingface_datasets'}")

Arquivos de áudio salvos em: /home/bruno-dev/fala_texto_share/projeto-fala-texto/backups/backup_2025-03-18_08-43-46/audio
Arquivo de transcrições salvo em: /home/bruno-dev/fala_texto_share/projeto-fala-texto/backups/backup_2025-03-18_08-43-46/transcriptions.csv
Arquivos de áudio do teste salvos em: /home/bruno-dev/fala_texto_share/projeto-fala-texto/backups/backup_2025-03-18_08-43-46/test_audio
Arquivo de transcrições do teste salvo em: /home/bruno-dev/fala_texto_share/projeto-fala-texto/backups/backup_2025-03-18_08-43-46/test_transcriptions.csv
Backup completo salvo em: /home/bruno-dev/fala_texto_share/projeto-fala-texto/backups/backup_2025-03-18_08-43-46
Pasta 'huggingface_datasets' movida para: /home/bruno-dev/fala_texto_share/projeto-fala-texto/backups/backup_2025-03-18_08-43-46/huggingface_datasets
